# GRPO — checkpointed group-relative policy optimization (run on the A100)

Fresh Colab, **A100 + High-RAM strongly recommended**, then run top to bottom. Trains the P6 QLoRA policy with GRPO so free-running **greedy** behavioral fidelity climbs from ~0.159 toward/past the oracle ~0.42.

**Output streams LIVE** (CELL 3b), so you can watch the `[grpo][eval] step=… greedy_fidelity=…` lines and decide when to stop.

### ⚠ How to "cut the wire" (read before the full run)
- To **stop training**: press the cell's **Stop / Interrupt** button **ONCE**. That sends a clean SIGINT — the loop checkpoints `latest/` at the next boundary and exits. `best/` is already current (written on every guard-clean eval improvement). Press twice to force-kill (you still keep the last `latest/`, saved every `ckpt_every` steps).
- **Do NOT stop the Colab RUNTIME/VM to stop training** — `/content` is **ephemeral**, so killing the VM **deletes `best/`**. Interrupt the *cell*, then run the **GATE** and **PUBLISH** cells while the VM is still alive, and only then disconnect.
- Anytime/resumable: if the VM preempts, re-run the full-run cell with the same `--run-id` — it resumes from `latest/`.

> Prerequisite: this clones **`feat/grpo`** (already pushed). Change `BRANCH` if you merged it elsewhere.

In [ ]:
# CELL 1 — provision the runtime (clone, install, HF token, stage the corpus)
import os, pathlib, subprocess, sys
REPO = '/content/SLM'
BRANCH = 'feat/grpo'
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
if not os.environ.get('SLM_PROVISIONED'):
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                s = _l.strip()
                if s.startswith(name + '='):
                    return s.split('=', 1)[1].strip().strip('"').strip("'")
    return None
import getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = _env_token('HF_TOKEN') or getpass.getpass('HF_TOKEN (read, for staging + adapter): ').strip()
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'          # lowercase corpus (the Colab case trap)
if not pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file():
    print('corpus missing -> staging ~9.85GB from hf://datasets/ericrcwu/LUT_SLM ...')
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
else:
    print('corpus already staged at /content/slm')

In [ ]:
# CELL 2 — build base_resized + download the P6 two-stage adapter (policy init AND frozen ref)
import os, pathlib, subprocess, sys, json
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
D = pathlib.Path('models/base_resized')
if not ((D / 'vocab_resize_manifest.json').is_file() and (D / 'preprocessor_config.json').is_file()):
    print('building base_resized (loads base in fp32 on CPU ~12GB; A100/High-RAM recommended) ...')
    subprocess.run([sys.executable, '-m', 'sft.vocab_resize', '--out', 'models/base_resized'],
                   check=True, env={**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'})
vr = json.load(open(D / 'vocab_resize_manifest.json'))
print('base_resized OK; identity:', vr['tokenizer_version'])

from huggingface_hub import snapshot_download
ADAPTER_SUBFOLDER = 'p6_twostage_d0f9c744_smokefull'
snapshot_download(repo_id='ericrcwu/LUT_SLM_sft_adapters', allow_patterns=[ADAPTER_SUBFOLDER + '/*'],
                  local_dir='models/sft_adapters', token=os.environ.get('HF_TOKEN'))
INIT_ADAPTER = 'models/sft_adapters/' + ADAPTER_SUBFOLDER
am = json.load(open(pathlib.Path(INIT_ADAPTER) / 'adapter_manifest.json'))
assert am['tokenizer_version'] == vr['tokenizer_version'], 'adapter/tokenizer identity mismatch'
assert pathlib.Path(INIT_ADAPTER, 'adapter_model.safetensors').is_file(), INIT_ADAPTER
print('P6 init adapter ready:', INIT_ADAPTER, '| step:', am.get('adapter_step'))

In [ ]:
# CELL 3 — GPU + stack sanity + off-GPU unit tests
import torch, transformers, bitsandbytes as bnb, peft, accelerate, qwen_vl_utils
from transformers import Qwen2_5_VLForConditionalGeneration
assert torch.cuda.is_available(), 'no CUDA — wrong runtime (need an A100)'
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| peft', peft.__version__, '| gpu', torch.cuda.get_device_name(0))
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pytest', '-q',
                    'tests/test_grpo_reward.py', 'tests/test_rollout.py',
                    'tests/test_grpo_loss.py', 'tests/test_grpo_train.py'],
                   capture_output=True, text=True)
print(r.stdout[-1500:]); print('unit tests', 'PASS' if r.returncode == 0 else 'FAIL')

In [ ]:
# CELL 3b — LIVE-streaming runners (so you SEE progress and can cut the wire cleanly)
import subprocess, sys, os, signal

def run_grpo(extra_args):
    """Run `sft.grpo_train` streaming output LIVE. Press the cell Stop/Interrupt button ONCE to send a
    clean SIGINT: the loop checkpoints `latest/` at the next boundary and exits (`best/` is already
    current). Returns the process exit code."""
    env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm', 'PYTHONUNBUFFERED': '1'}
    cmd = [sys.executable, '-u', '-m', 'sft.grpo_train'] + [str(a) for a in extra_args]
    print('>', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
        return proc.wait()
    except KeyboardInterrupt:
        print('\n[notebook] interrupt -> SIGINT to trainer; waiting for a clean checkpoint ...', flush=True)
        proc.send_signal(signal.SIGINT)
        try:
            for line in proc.stdout:
                print(line, end='', flush=True)
            return proc.wait(timeout=300)
        except Exception:
            proc.terminate(); return proc.wait()

def run_cmd(cmd, capture=False):
    """Stream any subprocess LIVE; optionally also capture stdout (to parse a summary line)."""
    env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm', 'PYTHONUNBUFFERED': '1'}
    print('>', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    buf = []
    for line in proc.stdout:
        print(line, end='', flush=True)
        if capture: buf.append(line)
    rc = proc.wait()
    return (rc, ''.join(buf)) if capture else rc

print('runners ready: run_grpo([...]) and run_cmd([...])')

## SMOKE — 4 steps, frequent ckpt/eval (acceptance B)

Expect `[grpo][OK]`, `latest/` + `best/` + `eval_log.jsonl`. `--eval-every 2 --ckpt-every 2` exercise the periodic-eval / BEST / atomic-`latest` paths in the tiny run.

In [ ]:
# CELL 4 — SMOKE run (live)
import pathlib
rc = run_grpo(['--config', 'configs/candidate_grpo.json', '--resized-model', 'models/base_resized',
               '--run-id', 'grpo_smoke', '--total-steps', 4, '--prompts-per-round', 2,
               '--eval-every', 2, '--ckpt-every', 2, '--eval-limit', 16])
run = pathlib.Path('models/sft_adapters/grpo_smoke')
print('\nrc', rc, '| latest', (run/'latest').is_dir(), '| best', (run/'best').is_dir(),
      '| eval_log', (run/'eval_log.jsonl').is_file())
assert rc == 0 and (run/'latest').is_dir(), 'smoke failed — scroll up'

## RESUME check — re-run the SAME run-id with a higher step budget (acceptance B, resume)

Reloads `latest/`, prints `[grpo] RESUME step=4 …`, continues to step 8. (A real preemption resumes the same way.)

In [ ]:
# CELL 5 — RESUME the smoke (live)
out = []
rc = run_cmd([__import__('sys').executable, '-u', '-m', 'sft.grpo_train',
              '--config', 'configs/candidate_grpo.json', '--resized-model', 'models/base_resized',
              '--run-id', 'grpo_smoke', '--total-steps', '8', '--prompts-per-round', '2',
              '--eval-every', '2', '--ckpt-every', '2', '--eval-limit', '16'], capture=True)
rc, text = rc
assert 'RESUME' in text, 'expected a [grpo] RESUME line (resume from latest/ did not trigger)'
print('\nRESUME OK — step continued from the smoke checkpoint.')

## FULL RUN — the real GRPO run (acceptance C)

Config defaults: `total_steps=500`, `group_size=8`, `eval_every=20`, `ckpt_every=20`. Watch the live `[grpo][eval]` lines.

**When to cut the wire:** stop when greedy fidelity has clearly moved up and plateaued (or an anti-hacking guard starts trending red) — press **Stop ONCE**. Then run the GATE + PUBLISH cells (below) *before* disconnecting the VM. If it OOMs, re-run with `'--prompts-per-round', 4`.

In [ ]:
# CELL 6 — FULL RUN (live; resumable — re-run this exact cell after any preemption)
rc = run_grpo(['--config', 'configs/candidate_grpo.json', '--resized-model', 'models/base_resized',
               '--run-id', 'grpo_r1'])
print('\nfull-run exit code:', rc)

In [ ]:
# CELL 6b — snapshot the eval log any time (also nice to run in a 2nd cell during CELL 6)
import json, pathlib
log = pathlib.Path('models/sft_adapters/grpo_r1/eval_log.jsonl')
if log.is_file():
    for line in log.read_text().splitlines():
        r = json.loads(line)
        print('step %4s | greedy_fid=%s collapse=%s degen=%s entropy=%s dE=%s kl=%s reward=%s %s' % (
            r.get('step'), r.get('behavioral_fidelity_mean'), r.get('collapse_rate'),
            r.get('degenerate_rate'), r.get('code_entropy_norm_mean'), r.get('decoded_delta_e_mean'),
            r.get('kl_to_ref'), r.get('reward_mean'), '<-- BEST' if r.get('is_best') else ''))
else:
    print('no eval_log yet')

## GATE — score `best/` on the full holdout (the real acceptance)

**Pass = free-running GREEDY `behavioral_fidelity_mean` beats 0.159 and moves toward/past ~0.42**, guards healthy, `oracle@32` not regressing.

In [ ]:
# CELL 7 — GATE: free-running greedy (+sample) behavioral fidelity on best/
import sys, json
BEST = 'models/sft_adapters/grpo_r1/best'
rc, out = run_cmd([sys.executable, '-u', '-m', 'sft.score_tokens',
                   '--config', 'configs/candidate_grpo.json', '--resized-model', 'models/base_resized',
                   '--adapter', BEST, '--behavioral-sampling', 'both', '--behavioral-limit', '0'],
                  capture=True)
summary = next((json.loads(l)['score_summary'] for l in out.splitlines()
                if l.strip().startswith('{') and 'score_summary' in l), None)
if summary:
    g = summary.get('behavioral') or {}
    print('\n================= GATE =================')
    print('GREEDY behavioral_fidelity_mean =', g.get('behavioral_fidelity_mean'),
          '(baseline greedy 0.159; best-of-N / oracle ~0.42)')
    print('  collapse=%s degenerate=%s deltaE=%s entropy=%s' % (
        g.get('collapse_rate'), g.get('degenerate_rate'),
        g.get('decoded_delta_e_mean'), g.get('code_entropy_norm_mean')))

In [ ]:
# CELL 8 — COVERAGE: oracle@N on best/ (must NOT regress) + best-of-N head-to-head
import sys
BEST = 'models/sft_adapters/grpo_r1/best'
run_cmd([sys.executable, '-u', '-m', 'eval.oracle_at_n',
         '--config', 'configs/candidate_grpo.json', '--resized-model', 'models/base_resized',
         '--adapter', BEST, '--limit', '0', '--n', '32', '--chunk', '16', '--temperatures', '0.7,1.0'])
run_cmd([sys.executable, '-u', '-m', 'eval.best_of_n', '--config', 'configs/candidate_grpo.json',
         '--adapter', BEST, '--limit', '0', '--n', '16', '--temperature', '1.0'])

## PUBLISH — persist `best/` to HF (do this BEFORE disconnecting the VM!)

`/content` is ephemeral. This uploads `best/` to the HF adapters repo as `grpo_r1_best` so it survives — and so Modal can pull it.

In [ ]:
# CELL 9 — upload best/ to HF (needs a WRITE token, e.g. SLM_Alpha_Write)
import os, getpass, pathlib
from huggingface_hub import upload_folder
RUN_ID = 'grpo_r1'
best = pathlib.Path('models/sft_adapters', RUN_ID, 'best')
assert best.is_dir(), f'no best/ at {best} — run the training first'
if not os.environ.get('HF_WRITE_TOKEN'):
    os.environ['HF_WRITE_TOKEN'] = getpass.getpass('HF_WRITE_TOKEN (write scope): ').strip()
DEST = f'{RUN_ID}_best'
upload_folder(repo_id='ericrcwu/LUT_SLM_sft_adapters', folder_path=str(best), path_in_repo=DEST,
              token=os.environ['HF_WRITE_TOKEN'])
print('published -> hf://ericrcwu/LUT_SLM_sft_adapters/' + DEST)
print('Now point Modal at it (see the PUBLISH-TO-MODAL cell / markdown below).')

## PUBLISH TO MODAL — serve the GRPO adapter

Modal's `setup_weights` pulls the generator adapter from HF (`ADAPTER_REPO` / `ADAPTER_SUBDIR` in `deploy/modal_app.py`). To serve the GRPO `best/` you just uploaded (`grpo_r1_best`):

**1. Point Modal at the new adapter.** In `deploy/modal_app.py` set:
```python
ADAPTER_SUBDIR = "grpo_r1_best"   # was p6_twostage_d0f9c744_smokefull
```
(The GRPO adapter is architecturally identical to P6 — same base, resized vocab, target modules — so it's a drop-in; the cached `base_resized` Volume still applies.)

**2. Refresh the Volume + redeploy** (run where the `modal` CLI is authed — your laptop is simplest; needs `modal secret create huggingface HF_TOKEN=…` once):
```bash
modal run    deploy/modal_app.py::setup_weights   # re-pulls the adapter into the slm-weights Volume
modal deploy deploy/modal_app.py                  # prints the public https://…modal.run URL
```

You can also run those two commands from a Colab cell (`!pip install modal` → `!modal token set …` → `!modal run …`), but a locally-authed `modal` is the cleaner path. Ask me and I can make `ADAPTER_SUBDIR` an env var (`SLM_ADAPTER_SUBDIR`) so no code edit is needed, and add a turnkey Colab Modal cell.